# Simulation study of the rule templates

## Purpose

The simulation checks whether a method detects a co-movement phenomenon when it is present, and how often it reports one when it is absent. Because we plant each phenomenon ourselves, the right answer is known. The same simulated data are given to two evaluators:

1. **the rule pipeline** (the coauthors' code) for every template (T1b, T2, T3b, T4, T7);
2. **the econometric test** of the same phenomenon (`mc_econ.py`).

Sample length is an explicit dimension, so we can tell whether a null result comes from the method or from short samples.

## Data-generating process (`dgp.py`)

**Marginals.** Pairs of daily log returns: X plays the S&P 500, Y the second asset.
- X follows a GJR-GARCH(1,1) with Student-t innovations (volatility rises more after falls).
- Y follows a GARCH(1,1) with Student-t innovations.
- Both are calibrated by `calibrate.py` on the series used by the rules: the S&P 500 and COMEX gold of the July 2026 vintage (`GOLD_FUT`), 2000–2024, days on which both returns exist, with variance targeting. The estimates are written to `calibration.json`, which `dgp.py` reads.
- A quality check against the real data is in `check_dgp.py`.

**Link between the two series.** The standardised innovations are linked by a time-varying correlation ρ_t. The background correlation ρ0 is the *baseline* dimension; θ is the strength of the planted phenomenon (θ = 0: nothing planted).

| Template | Planted phenomenon | θ grid |
|---|---|---|
| T1b safe haven | ρ_t = ρ0 − θ on days when X's shock is in its lower 10% tail (Baur–Lucey definition) | 0, 0.15, 0.30, 0.45, 0.60 |
| T2 contagion | ρ_t = ρ0 + θ when X is in a high-volatility window. Under the null, X's variance still rises in crises (the Forbes–Rigobon trap) | 0, 0.1, 0.2, 0.3, 0.4 |
| T3b decoupling | ρ_t = ρ0 (1 − θ) when both series are calm | 0, 0.25, 0.5, 0.75, 1 |
| T4 lead in stress | Y loads θ on X's shock 5 days earlier when X is in a high-volatility window | 0, 0.05, 0.10, 0.15, 0.20 |
| T7 macro | ρ_t = ρ0 + θ when inflation is high, defined as in the rules: monthly m/m CPI inflation follows an AR(1) calibrated on 1985–2024 (`calibrate.py`); it is smoothed with a centred 3-month moving average, and high means above the 75th percentile of the training months; the state of month m holds on its days (no publication lag) | 0, 0.1, 0.2, 0.3, 0.4 |

**States.** Computed from the 50-day window ending the previous day, with the predicate definitions of the rules. **highVol = at least 30% of the window above the 90th percentile of |r_X|** (since 12 Nov 2026; it was 25% before). The threshold affects only the planted T2 and T4 samples; samples with θ = 0 are identical under both thresholds.

**Other dimensions.**

| Dimension | Values | Note |
|---|---|---|
| Baseline ρ0 | 0, 0.3, 0.6 | T3b uses only 0.3 and 0.6, since there is nothing to decouple at 0 |
| Sample length | 2,520, 5,040 and 10,080 trading days (10, 20, 40 years) | Split 70% training, 30% test; 10 years reproduces the real split (756 test days) |
| Replications | 200 per configuration | |

Each configuration has a deterministic seed (`mc_econ.config_seed`), so the same datasets are produced on every run.

## Running the simulation, and when it does not start

`python main.py` (or `python main.py simulation export`) **does not start the simulation if simulated datasets already exist** in `simulation/datasets/`: the calibration, the Monte Carlo and the export are skipped, and only the summaries and plots are redrawn from the stored results (`results_econ.csv`). This protects the datasets the rules were run on.

To regenerate it anyway:
- in `main.py`, uncomment the line `# FORCE_SIMULATION = True` (just below `FORCE_SIMULATION = False`), run, and comment it again afterwards;
- or pass the flag: `python main.py --force-simulation` (or `python main.py simulation export --sets core long --force-simulation`);
- or, when `main.py` is run from a terminal, answer `y` to the question it asks when it finds existing datasets (the default, Enter, keeps them).

Regenerating recalibrates the model, reruns the Monte Carlo from scratch (the old results are kept as `results_econ_previous.csv`) and overwrites the datasets. It takes about an hour. After a regeneration, the rules must be re-run on all the datasets.

## Datasets for the rule pipeline (`export_datasets.py`)

`python main.py export --sets core long` (or `python simulation/export_datasets.py core long`) writes one compressed CSV per configuration, with all 200 replications, to `datasets/<set>/`:

- columns `rep`, `date` (placeholder business-day calendar), `X`, `Y` (daily log returns), `split` (`train` / `test`), `inflationHigh` (T7 only);
- `configurations.csv`: one row per file with template, θ, ρ0, length, seed, train and test days, the planted phenomenon and the detection rate of the econometric test on the same samples (the benchmark for the rules).

| Set | Content | Size |
|---|---|---|
| `core` | One configuration per template, at 10 years, with and without the phenomenon: T1b ρ0 0.3 θ 0.45; T2 ρ0 0.3 θ 0.4; T3b ρ0 0.6 θ 1.0; T4 ρ0 0.3 θ 0.2; T7 ρ0 0.3 θ 0.2 | 10 files, about 58 MB |
| `long` | The same configurations at 20 and 40 years | 20 files, about 331 MB (regenerate rather than commit) |
| `grid` | Every configuration of the Monte Carlo | 210 files, about 2.8 GB |

The datasets are identical to those used by `mc_econ.py`, so rules and tests are compared on the same samples. **The datasets were regenerated on 12 Nov 2026** (calibration on the July gold vintage, inflation state of the rules, highVol at 30%): the rules must be re-run on all of them.

We report, for each configuration:
- **false alarms (size):** the detection rate at θ = 0 (target 5%), detection meaning significant;
- **power:** the detection rate at θ > 0;
- **testable share:** the share of replications with enough trigger days to be scored (unscorable replications count as not detected).

## Scripts and outputs

- `calibrate.py` estimates the marginals and the inflation process and writes `calibration.json`.
- `check_dgp.py` compares simulated and real moments and trigger frequencies (printed only).
- `mc_econ.py` runs the Monte Carlo of the econometric tests and writes `results_econ.csv` (resumable: only configurations missing from the file are run).
- `summarize_econ.py` writes `figures_tables_repository/fig_sim_power_econ.pdf` and `figures_tables_repository/tab_sim_econ.tex`.
- `plot_sim_series.py` plots single simulated samples (fixed seeds) in `figures_tables_repository/`:
  - `fig_sim_vs_real.pdf`: real S&P 500 and gold (last 10 years to 2024) next to one simulated null sample;
  - `fig_sim_series_<T>.pdf`: one 10-year sample per template (ρ0 = 0.3), with returns, the 50-day rolling correlation against the planted one, and the trigger state shaded.

  Return panels use a fixed range (±15% for X, ±12% for Y); days beyond it are marked at the edge. With Student-t innovations a few replications contain days larger than any in the real data; the median of the largest daily move over 25 simulated years matches the real one (S&P 13.0% vs 12.8%, gold 10.0% vs 9.8%).

## Main results of the econometric tests

Detection rates at 10 / 40 years for the `core` configurations (regenerated 12 Nov 2026), false alarms (θ = 0) and power:

| Template | False alarms | Power |
|---|---|---|
| T1b (Baur–Lucey) | 0.0 / 0.0 | 0.41 / 0.63 |
| T2 (Forbes–Rigobon) | 0.03 / 0.035 | 0.08 / 0.10 |
| T3b (calm-day correlation) | 0.015 / 0.0 | 0.415 / 0.355 |
| T4 (predictive regression) | 0.05 / 0.08 | 0.365 / 0.66 |
| T7 (conditional correlation) | 0.05 / 0.035 | 0.73 / 1.0 |